In [1]:
import psycopg
import pandas as pd
from fmiopendata.wfs import download_stored_query
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
params = "p_sea,t2m,rh,ws_10min,wd_10min,r_1h,n_man,vis"

In [4]:
conn = psycopg.connect(
    host="localhost",
    dbname="weather",
    user=os.getenv("PSQL_USER"),
    password=os.getenv("PSQL_PASSWORD")
)
cursor = conn.cursor()

In [5]:
forecasts_df = pd.read_sql(
    """SELECT forecasts.forecast_time, forecasts.fmisid, stations.name,
    forecasts.air_pressure, forecasts.air_temperature, forecasts.humidity,
    forecasts.wind_speed, forecasts.wind_direction, forecasts.precipitation_amount,
    forecasts.total_cloud_cover, forecasts.visibility
    FROM forecasts
    INNER JOIN stations ON forecasts.fmisid=stations.fmisid
    ;""",
    conn
)

/tmp/ipykernel_42450/2526240131.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  forecasts_df = pd.read_sql(


In [6]:
forecasts_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 252 entries, 0 to 251
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   forecast_time         252 non-null    datetime64[us]
 1   fmisid                252 non-null    int64         
 2   name                  252 non-null    str           
 3   air_pressure          252 non-null    float64       
 4   air_temperature       252 non-null    float64       
 5   humidity              252 non-null    float64       
 6   wind_speed            252 non-null    float64       
 7   wind_direction        252 non-null    float64       
 8   precipitation_amount  252 non-null    float64       
 9   total_cloud_cover     252 non-null    float64       
 10  visibility            252 non-null    float64       
dtypes: datetime64[us](1), float64(8), int64(1), str(1)
memory usage: 21.8 KB


In [7]:
oldest_forecast_time = forecasts_df["forecast_time"].min()
newest_forecast_time = forecasts_df["forecast_time"].max()

In [8]:
obs = download_stored_query(
    "fmi::observations::weather::multipointcoverage",
    args=[
        "fmisid=101042", # These are the IDs for each weather station
        "fmisid=101030",
        "fmisid=101039",
        "fmisid=101023",
        "fmisid=101022",
        "fmisid=100683",
        "fmisid=105392",
        "fmisid=100996",
        "fmisid=100997",
        "fmisid=108020",
        "fmisid=100969",
        "fmisid=100953",
        "fmisid=100932",
        "fmisid=100946",
        "fmisid=100945",
        "fmisid=100908",
        "fmisid=100924",
        "fmisid=100947",
        "fmisid=100934",
        "fmisid=100928",
        "fmisid=151048",
        "fmisid=100909",
        "fmisid=151029",
        "fmisid=100919",
        "fmisid=101059",
        "fmisid=101061",
        "fmisid=101267",
        "fmisid=101256",
        "fmisid=101479",
        "fmisid=101481",
        "fmisid=101485",
        "fmisid=101464",
        "fmisid=101660",
        "fmisid=101675",
        "fmisid=101661",
        "fmisid=101673",
        "fmisid=101775",
        "fmisid=101785",
        "fmisid=101784",
        "fmisid=101794",
        "fmisid=101783",
        "fmisid=101846",
        "starttime=" + str(oldest_forecast_time),
        "endtime=" + str(newest_forecast_time),
        "parameters=" + params,
        "timestep=60"
    ]
)

In [9]:
obs.data.keys()

dict_keys([datetime.datetime(2026, 4, 28, 11, 0), datetime.datetime(2026, 4, 28, 12, 0), datetime.datetime(2026, 4, 28, 13, 0), datetime.datetime(2026, 4, 28, 14, 0), datetime.datetime(2026, 4, 28, 15, 0), datetime.datetime(2026, 4, 28, 16, 0), datetime.datetime(2026, 4, 28, 17, 0), datetime.datetime(2026, 4, 28, 18, 0), datetime.datetime(2026, 4, 28, 19, 0), datetime.datetime(2026, 4, 28, 20, 0), datetime.datetime(2026, 4, 28, 21, 0), datetime.datetime(2026, 4, 28, 22, 0), datetime.datetime(2026, 4, 28, 23, 0), datetime.datetime(2026, 4, 29, 0, 0), datetime.datetime(2026, 4, 29, 1, 0), datetime.datetime(2026, 4, 29, 2, 0), datetime.datetime(2026, 4, 29, 3, 0), datetime.datetime(2026, 4, 29, 4, 0), datetime.datetime(2026, 4, 29, 5, 0), datetime.datetime(2026, 4, 29, 6, 0), datetime.datetime(2026, 4, 29, 7, 0), datetime.datetime(2026, 4, 29, 8, 0), datetime.datetime(2026, 4, 29, 9, 0), datetime.datetime(2026, 4, 29, 10, 0), datetime.datetime(2026, 4, 29, 11, 0)])

In [10]:
rows = [
    {"time": ts, "station": station, **{p: v["value"] for p, v in params.items()}}
    for ts, stations in obs.data.items()
    for station, params in stations.items()
]

obs_df = pd.DataFrame(rows).sort_values(["station", "time"])

In [11]:
obs_df.isna().sum()

time                       0
station                    0
Pressure (msl)           152
Air temperature            2
Relative humidity          2
Wind speed                52
Wind direction            52
Precipitation amount     800
Cloud amount             627
Horizontal visibility    577
dtype: int64

In [12]:
forecasts_df.isna().sum()

forecast_time           0
fmisid                  0
name                    0
air_pressure            0
air_temperature         0
humidity                0
wind_speed              0
wind_direction          0
precipitation_amount    0
total_cloud_cover       0
visibility              0
dtype: int64

In [13]:
merged_df = obs_df.merge(
    forecasts_df, 
    left_on=["time", "station"],
    right_on=["forecast_time", "name"],
    suffixes=("_obs", "_fcst")
)

In [14]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 246 entries, 0 to 245
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   time                   246 non-null    datetime64[us]
 1   station                246 non-null    str           
 2   Pressure (msl)         208 non-null    float64       
 3   Air temperature        244 non-null    float64       
 4   Relative humidity      244 non-null    float64       
 5   Wind speed             232 non-null    float64       
 6   Wind direction         232 non-null    float64       
 7   Precipitation amount   60 non-null     float64       
 8   Cloud amount           94 non-null     float64       
 9   Horizontal visibility  106 non-null    float64       
 10  forecast_time          246 non-null    datetime64[us]
 11  fmisid                 246 non-null    int64         
 12  name                   246 non-null    str           
 13  air_pressure    

In [15]:
merged_df

,time,station,Pressure (msl),Air temperature,Relative humidity,Wind speed,Wind direction,Precipitation amount,Cloud amount,Horizontal visibility,...,fmisid,name,air_pressure,air_temperature,humidity,wind_speed,wind_direction,precipitation_amount,total_cloud_cover,visibility
0,2026-04-28 11:00:00,Hailuoto Marjaniemi,1023.3,3.2,64.0,11.3,354.0,NaN,7.0,20000.0,...,101784,Hailuoto Marjaniemi,1023.4,2.6,71.3,6.74,350.0,0.0,7.5,60594.3
1,2026-04-28 12:00:00,Hailuoto Marjaniemi,1023.5,3.4,62.0,10.2,353.0,NaN,4.0,20000.0,...,101784,Hailuoto Marjaniemi,1023.5,2.8,69.9,6.45,345.0,0.0,5.1,62940.0
2,2026-04-28 13:00:00,Hailuoto Marjaniemi,1023.6,3.8,62.0,10.3,2.0,NaN,6.0,20000.0,...,101784,Hailuoto Marjaniemi,1023.8,2.8,70.6,6.50,343.0,0.0,26.0,61863.4
3,2026-04-28 14:00:00,Hailuoto Marjaniemi,1023.9,3.4,67.0,8.7,360.0,NaN,7.0,20000.0,...,101784,Hailuoto Marjaniemi,1023.8,3.0,69.6,6.29,348.0,0.0,9.2,63565.3
4,2026-04-29 09:00:00,Hailuoto Marjaniemi,1024.9,2.5,52.0,3.8,322.0,NaN,0.0,20000.0,...,101784,Hailuoto Marjaniemi,1024.9,2.3,69.7,4.08,287.0,0.0,0.0,63231.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241,2026-04-28 12:00:00,Vaasa Klemettilä,1025.9,4.7,39.0,10.5,17.0,0.0,0.0,43246.0,...,101485,Vaasa Klemettilä,1025.8,6.0,42.3,7.66,5.0,0.0,0.0,59915.3
242,2026-04-28 13:00:00,Vaasa Klemettilä,1025.7,5.1,41.0,9.1,33.0,0.0,0.0,44860.0,...,101485,Vaasa Klemettilä,1025.8,6.6,39.5,7.93,17.0,0.0,0.0,61808.1
243,2026-04-28 14:00:00,Vaasa Klemettilä,1025.8,5.4,41.0,7.4,22.0,0.0,0.0,45470.0,...,101485,Vaasa Klemettilä,1025.5,6.7,41.1,6.72,20.0,0.0,0.0,60765.3
244,2026-04-29 09:00:00,Vaasa Klemettilä,1026.2,6.6,40.0,3.3,275.0,0.0,0.0,46945.0,...,101485,Vaasa Klemettilä,1026.4,6.9,41.5,5.01,272.0,0.0,68.1,60805.6


In [16]:
def circular_diff(a, b):
    return ((a - b + 180) % 360) - 180

In [17]:
#  MAE = Mean Absolute Error
MAE_air_pressure = (merged_df["Pressure (msl)"] - merged_df["air_pressure"]).abs().mean()
MAE_air_temperature = (merged_df["Air temperature"] - merged_df["air_temperature"]).abs().mean()
MAE_humidity = (merged_df["Relative humidity"] - merged_df["humidity"]).abs().mean()
MAE_wind_speed = (merged_df["Wind speed"] - merged_df["wind_speed"]).abs().mean()
MAE_wind_direction = circular_diff(merged_df["Wind direction"], merged_df["wind_direction"]).abs().mean()
MAE_precip_amount = (merged_df["Precipitation amount"] - merged_df["precipitation_amount"]).abs().mean()
MAE_visibility = (merged_df["Horizontal visibility"] - merged_df["visibility"]).abs().mean()

In [18]:
print(f"MAE for air pressure (msl): {round(MAE_air_pressure, 2)} hPa. Calculated from {merged_df["Pressure (msl)"].count()} rows.")
print(f"\nMAE for air temperature: {round(MAE_air_temperature, 2)} celsius. Calculated from {merged_df["Air temperature"].count()} rows.")
print(f"\nMAE for relative humidity: {round(MAE_humidity, 2)} %. Calculated from {merged_df["Relative humidity"].count()} rows.")
print(f"\nMAE for wind speed: {round(MAE_wind_speed, 2)} m/s. Calculated from {merged_df["Wind speed"].count()} rows.")
print(f"\nMAE for wind wind direction: {round(MAE_wind_direction, 2)} degrees. Calculated from {merged_df["Wind direction"].count()} rows.")
print(f"\nMAE for precipitation amount: {MAE_precip_amount} mm. Calculated from {merged_df["Precipitation amount"].count()} rows.")
print(f"\nMAE for horizontal visibility: {MAE_visibility} meters. Calculated from {merged_df["Horizontal visibility"].count()} rows.")

MAE for air pressure (msl): 0.37 hPa. Calculated from 208 rows.

MAE for air temperature: 0.81 celsius. Calculated from 244 rows.

MAE for relative humidity: 7.94 %. Calculated from 244 rows.

MAE for wind speed: 1.64 m/s. Calculated from 232 rows.

MAE for wind wind direction: 13.51 degrees. Calculated from 232 rows.

MAE for precipitation amount: 0.0 mm. Calculated from 60 rows.

MAE for horizontal visibility: 34969.38962264151 meters. Calculated from 106 rows.
